# 00 · Warm-up: what's wrong with this notebook?

| Session | Time | Previous | Next |
| --- | --- | --- | --- |
| Day 1 | about 30 minutes live | [P0 · Python refresher (pre-work)](P0_python_refresher.ipynb) | [01 · Classes and objects](01_classes_and_objects.ipynb) |

Below is a churn-prediction notebook of the kind that gets written under time pressure, when the priority is "make it work" rather than "make it last". It loads real customer data from a telecom company, trains a model and prints a report.

And it works. Run it top to bottom and you get sensible numbers.

But it hides problems, one of them serious enough to send the wrong customers to the retention team. Over the next two days you will learn the tools to fix every one of them. Today, you only need to find them.

> **Learning goals.** By the end you can:
>
> - point out concrete places where notebook code becomes hard to reuse, test or run in production
> - describe a bug that produces wrong results without any error message
> - explain your findings in plain words, to a partner or to the group

**How to work.** First read and run the notebook (about 5 minutes). Then investigate it with the exercises, alone or with a partner (about 15 minutes). In pairs, one person types and the other reads along and asks questions; online, the person typing shares their screen. At the end we collect the problems as a group, in the room or in the chat.

**In this notebook**

1. Hands-on A · Run the notebook and meet the mystery
2. Hands-on B · Investigate
3. Hands-on C · Collect the problems

---

## Hands-on A · Run the notebook and meet the mystery

The business question: which customers are likely to cancel their contract next month, so the retention team can contact them with an offer?

> **Reading help: pandas in cells 1 to 7.** You do not need to understand every line to find the problems, but here is what the less obvious ones do:
>
> - `pd.to_numeric(column, errors="coerce")` turns text into numbers; text that is not a number becomes `NaN` (missing).
> - `df2.dropna()` returns the table without the rows that contain a missing value.
> - `le.fit_transform(column)` (a scikit-learn `LabelEncoder`) replaces each text value by a whole number: it sorts the different values it sees and numbers them 0, 1, 2, ...
> - `df2[features]` keeps only the listed columns; `X` holds the model inputs, `y` the 0/1 target.

In [ ]:
# --- Cell 1: load data ---
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("../data/raw/telco_churn.csv")
print(df.shape)
df.head()

In [ ]:
# --- Cell 2: quick look ---
df.info()

In [ ]:
# --- Cell 3: TotalCharges is text for some reason, fix it ---
df2 = df.copy()
df2["TotalCharges"] = pd.to_numeric(df2["TotalCharges"], errors="coerce")
df2 = df2.dropna()
print(df2.shape)

In [ ]:
# --- Cell 4: encode ---
le = LabelEncoder()
df2["Contract"] = le.fit_transform(df2["Contract"])
df2["InternetService"] = le.fit_transform(df2["InternetService"])
df2["PaymentMethod"] = le.fit_transform(df2["PaymentMethod"])
df2["PaperlessBilling"] = le.fit_transform(df2["PaperlessBilling"])
df2["Churn"] = le.fit_transform(df2["Churn"])
# there are more columns but let's just use these for now

In [ ]:
# --- Cell 5: features and target ---
features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "SeniorCitizen",
    "Contract",
    "InternetService",
    "PaymentMethod",
    "PaperlessBilling",
]
X = df2[features]
y = df2["Churn"]

In [ ]:
# --- Cell 6: split and train ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=200, min_samples_leaf=10, random_state=42)
model.fit(X_train, y_train)

In [ ]:
# --- Cell 7: evaluate ---
preds = model.predict(X_test)
print(classification_report(y_test, preds))

A recall of about 0.5 on the churners, an accuracy near 0.8. Not bad for a first model. The notebook goes into the team's shared folder.

### One month later

The CRM system exports the customers whose **contract ends next month**. Only one-year and two-year contracts have an end date, so the export contains only those. The retention team wants to know whom to call. A colleague adds two cells:

> **Reading help: pandas and NumPy in cells 8 to 10.**
>
> - `raw[raw["Contract"] != "Month-to-month"]` keeps the rows whose contract is not month-to-month; `.sample(200, random_state=7)` then picks 200 of them at random (the same 200 every time).
> - `model.predict_proba(X)` returns one row per customer with two probabilities, "stays" and "churns"; `[:, 1]` takes the second column, the churn risk of every customer.
> - `(risk >= 0.3).sum()` counts the customers whose risk is at least 30 %: the comparison gives `True`/`False` per customer, and `.sum()` counts the `True`s.
> - `df2.loc[new.index, features]` takes, from the table prepared in cells 3 and 4, the rows of the same customers and the model's columns.
> - `np.abs(a - b).max()` is the largest difference between two arrays of risks.

In [ ]:
# --- Cell 8: score next month's renewals ---
raw = pd.read_csv("../data/raw/telco_churn.csv")
renewals = raw[raw["Contract"] != "Month-to-month"].sample(
    200, random_state=7
)  # stands in for the CRM export

new = renewals.copy()
new["TotalCharges"] = pd.to_numeric(new["TotalCharges"], errors="coerce")
new = new.dropna()
new["Contract"] = le.fit_transform(new["Contract"])
new["InternetService"] = le.fit_transform(new["InternetService"])
new["PaymentMethod"] = le.fit_transform(new["PaymentMethod"])
new["PaperlessBilling"] = le.fit_transform(new["PaperlessBilling"])

risk = model.predict_proba(new[features])[:, 1]
print(f"Customers to call (risk above 30 %): {(risk >= 0.3).sum()}")
print(f"Average churn risk in this batch: {risk.mean():.1%}")

The team calls those customers. Nothing crashed, no warning was printed.

### Something does not add up

These 200 customers are also in our historical table, which we prepared in cells 3 and 4. Let's score the very same rows from there:

In [ ]:
# --- Cell 9: sanity check, same customers, same model ---
same_rows = df2.loc[new.index, features]
risk_check = model.predict_proba(same_rows)[:, 1]
print(f"Customers to call (risk above 30 %): {(risk_check >= 0.3).sum()}")
print(f"Customers whose call decision differs: {((risk >= 0.3) != (risk_check >= 0.3)).sum()}")
print(f"Average churn risk: {risk_check.mean():.1%}")
print(f"Largest difference for a single customer: {np.abs(risk - risk_check).max():.2f}")

Same model, same customers, different answers. The average risk roughly doubles, and the list of customers to call is about four times longer. Most of the extra calls go to customers the model, scored correctly, considers safe.

And here is the part that makes it hard to catch: try it with a **random** sample of customers instead of the renewals.

In [ ]:
# --- Cell 10: same procedure, random customers ---
batch = raw.sample(200, random_state=7)
b = batch.copy()
b["TotalCharges"] = pd.to_numeric(b["TotalCharges"], errors="coerce")
b = b.dropna()
for col in ["Contract", "InternetService", "PaymentMethod", "PaperlessBilling"]:
    b[col] = le.fit_transform(b[col])

diff = np.abs(
    model.predict_proba(b[features])[:, 1] - model.predict_proba(df2.loc[b.index, features])[:, 1]
)
print(f"Largest difference on a random batch: {diff.max():.2f}")

On a random batch everything matches. A quick test by the colleague would have passed.

**The mystery: why do the renewals get different scores, and why only them?** Don't look for the answer online yet. The exercises below help you collect evidence; we solve it together in notebook 04.

---

## Hands-on B · Investigate

Four small experiments. You only need the code that is already in this notebook: copy, change, run.

### Exercise B.1

*About 8 minutes, including the discussion.*

Which batches of customers get wrong scores? The cell below scores a batch the way cell 8 does, and the same customers the way cells 3 and 4 prepared them. Change the filter in the first line and run the cell again for each of these batches:

1. `raw[raw["Contract"] == "Two year"].head(100)`
2. `raw[raw["Contract"] == "Month-to-month"].head(100)`
3. `raw[raw["InternetService"] == "Fiber optic"].head(100)`
4. `raw[raw["PaymentMethod"] == "Electronic check"].head(100)`
5. `raw.sample(100, random_state=3)` (a random batch)

Note for each batch whether the two numbers of calls agree.

**What your code should do.** You only change the first line of the cell; everything else stays. `batch` must be a DataFrame of customers taken from `raw`, the untouched file. The cell then prints two numbers of calls (the same customers prepared in two ways) and the largest difference in risk for a single customer.

**pandas you need.**

- `raw[raw["Contract"] == "Two year"]` keeps only the rows where the condition is true.
- `.head(100)` takes the first 100 of those rows; `raw.sample(100, random_state=3)` takes 100 random rows (the same ones every time).

**Discuss.** What do the batches with different answers have in common, and what do the batches without differences have in common? Write down your hypothesis; notebook 04 will test it.

In [ ]:
# Exercise B.1: change the filter in the first line, run, compare
batch = raw[raw["Contract"] == "Two year"].head(100)

b = batch.copy()
b["TotalCharges"] = pd.to_numeric(b["TotalCharges"], errors="coerce")
b = b.dropna()
for col in ["Contract", "InternetService", "PaymentMethod", "PaperlessBilling"]:
    b[col] = le.fit_transform(b[col])

risk_new = model.predict_proba(b[features])[:, 1]
risk_hist = model.predict_proba(df2.loc[b.index, features])[:, 1]
print(f"Calls, scored like cell 8:         {(risk_new >= 0.3).sum()}")
print(f"Calls, scored from cells 3 and 4:  {(risk_hist >= 0.3).sum()}")
print(f"Largest difference for one customer: {abs(risk_new - risk_hist).max():.2f}")

<details><summary><b>Show what you should see</b></summary>

| Batch | Calls agree? |
| --- | --- |
| Two year only | no |
| Month-to-month only | yes |
| Fiber optic only | no |
| Electronic check only | no |
| Random | yes |

The interesting pair is "Two year only" (wrong) versus "Month-to-month only" (right): both batches contain a single contract type. Keep that in mind for notebook 04.
</details>

### Exercise B.2

*About 5 minutes, including the discussion.*

Cell 3 drops rows with `dropna()`. Find out which customers disappear: store the rows of `df` whose `TotalCharges` cannot be converted to a number in a DataFrame called `dropped`, then look at their `tenure`.

**What your code should do.**

- `dropped` is a DataFrame with the same columns as `df` and 11 rows: exactly the customers whose `TotalCharges` is not a number.
- Do not change `df` itself.

**pandas you need.**

- `pd.to_numeric(df["TotalCharges"], errors="coerce")` gives a column of numbers with `NaN` where the conversion failed.
- `.isna()` turns that into `True` (missing) or `False` per row.
- `df[mask]` keeps the rows where `mask` is `True`.

**Discuss.** Who are these customers? Should the model see them, and what should their `TotalCharges` be? Who in the company could confirm your answer?

In [ ]:
# Exercise B.2: your code here

dropped = None  # the rows of df that cell 3 removes

In [ ]:
# Check for exercise B.2: run after solving it
assert len(dropped) == 11
print(dropped[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]])

<details><summary><b>Show hint</b></summary>

`pd.to_numeric(df["TotalCharges"], errors="coerce")` gives `NaN` exactly where the conversion fails, and `.isna()` turns that into True/False. Use it as a filter on `df`.
</details>

<details><summary><b>Show solution</b></summary>

```python
dropped = df[pd.to_numeric(df["TotalCharges"], errors="coerce").isna()]
```

All 11 have `tenure == 0`: they joined this month and have not received a bill yet, so the export leaves `TotalCharges` blank. The correct value is 0, not "missing". Dropping them means the model never learns anything about brand-new customers.
</details>

### Exercise B.3
> If you finish early, try this one.

How much does the report depend on luck? Repeat the split and training of cell 6 with `random_state` 0, 1, 2, 3 and 4 for `train_test_split` (keep the model's own `random_state=42`) and print the recall on churners for each run. `sklearn.metrics.recall_score(y_test, preds)` computes it.

**What your code should do.**

- Run five times: split `X` and `y` with `train_test_split(X, y, test_size=0.2, random_state=seed)`, train a fresh model exactly as in cell 6, predict on the test part.
- Print one line per seed with the seed and the recall, e.g. `0 0.512`.
- Do not overwrite `X_train`, `X_test`, `model` and friends: use new names (`X_tr`, `m`, ...), so the rest of the notebook keeps working.

**Python and scikit-learn you need.**

- `for seed in [0, 1, 2, 3, 4]:` repeats the indented block once per value.
- `recall_score(y_true, y_pred)` returns one number between 0 and 1: the share of churners the model found.

**Discuss.** If recall moves by several points just because the split changed, what does that mean when a colleague says "my new feature improves recall by 0.02"? And what happens to the results of this notebook if someone runs its cells in a different order?

In [ ]:
# Exercise B.3: your code here


<details><summary><b>Show solution</b></summary>

```python
from sklearn.metrics import recall_score

for seed in [0, 1, 2, 3, 4]:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    m = RandomForestClassifier(n_estimators=200, min_samples_leaf=10, random_state=42)
    m.fit(X_tr, y_tr)
    print(seed, round(recall_score(y_te, m.predict(X_te)), 3))
```

Recall moves between about 0.46 and 0.53. A difference of 0.02 between two models can be pure luck of the split. Comparing models fairly needs the same split for both, or better, cross-validation.
</details>

### Exercise B.4
> **Challenge**

The obvious fix for the copy-pasted preprocessing is a function. Write `prepare(batch)` that performs the steps of cells 3 and 4 (convert `TotalCharges`, drop missing rows, encode the four text columns with `le`) and returns the prepared DataFrame. Then run the cell after it, which prepares both the full table and the renewals with your function. Does the bug disappear?

**What your code should do.**

- `prepare(batch)` takes a DataFrame in the format of the raw file (like `raw` or `renewals`) and returns a new, prepared DataFrame; it must not change `batch` itself.
- The result has numeric `TotalCharges`, no rows with missing values, and the four text columns `Contract`, `InternetService`, `PaymentMethod` and `PaperlessBilling` encoded with `le`, exactly as in cells 3 and 4.

**pandas you need.** `batch.copy()` (work on a copy), `pd.to_numeric(..., errors="coerce")`, `.dropna()` and `le.fit_transform(column)`, all used in cells 3 and 4.

**Discuss.** The duplication is gone. Are the wrong answers gone too? What would `prepare` have to remember between the call on the training data and the call on next month's file, and where could that memory live?

In [ ]:
# Exercise B.4: your code here


def prepare(batch): ...  # the steps of cells 3 and 4, returning the prepared DataFrame

In [ ]:
# Check for exercise B.4: run after solving it
train_ready = prepare(raw)
renewals_ready = prepare(renewals)
risk_fn = model.predict_proba(renewals_ready[features])[:, 1]
risk_ok = model.predict_proba(train_ready.loc[renewals_ready.index, features])[:, 1]
print(
    f"Calls with your function: {(risk_fn >= 0.3).sum()}, calls from the training preparation: {(risk_ok >= 0.3).sum()}"
)

<details><summary><b>Show solution</b></summary>

```python
def prepare(batch):
    out = batch.copy()
    out["TotalCharges"] = pd.to_numeric(out["TotalCharges"], errors="coerce")
    out = out.dropna()
    for col in ["Contract", "InternetService", "PaymentMethod", "PaperlessBilling"]:
        out[col] = le.fit_transform(out[col])
    return out
```

The output is still 17 against 4. A function removes the copy-paste, but it starts from zero on every call: it has nowhere to keep what it learned from the training data. Something that holds data *and* the code that works on it is exactly what a class is (notebook 01), and notebook 04 builds the encoder that fixes this bug.
</details>

---
## Hands-on C · Collect the problems

Alone or with your partner, collect every place where this notebook would hurt you or your team. Write them down in the cell below (online: also post your top three in the chat). Some questions to guide you:

- **Reuse**: what would you have to copy-paste to train on a new data extract, or to try a second model?
- **Change**: what breaks if the CRM renames a column, or adds a new payment method?
- **Readability**: what do `df2`, `le` and the numbers `200`, `10`, `0.3` mean, and who decides them?
- **Correctness**: what happened to the 11 rows dropped in cell 3? Who are those customers? (Hint: look at their `tenure`.)
- **Testing**: how would you prove that cell 8 is correct before the retention team acts on it?
- **Production**: could this run unattended every month? What would happen if the file were missing?

*Your notes:*

-
-
-

<details><summary><b>Some of the problems trainers usually hear (open after collecting your own)</b></summary>

- Preprocessing is copy-pasted between training (cells 3 and 4) and scoring (cell 8). Any change must be made twice, and it already diverged.
- One `LabelEncoder` object is re-fitted again and again, each time on different data. What it "knows" at the end depends on the last column it saw.
- Magic numbers (`0.2`, `42`, `200`, `10`, `0.3`) and a hard-coded path, with no explanation of why.
- `dropna()` silently removes 11 customers. They have `tenure == 0`: they joined this month and have not been billed yet. Their `TotalCharges` should be 0, not "missing", and dropping them means the model never sees brand-new customers.
- Variables like `df`, `df2`, `X`, `new`, `b` are overwritten as the notebook runs. Running the cells in a different order gives different results.
- There is nothing to test: no functions, no classes, no single place that defines "how a customer is prepared for the model".
- Nothing tells you when something goes wrong: no validation of the input columns, no logging, and the scoring bug produced no error at all.
- The trained model and its preprocessing live only in memory. Next month someone has to re-run the whole notebook.

</details>

---

## Where we fix each problem

| Problem | Tool | Notebook |
|---|---|---|
| Loose variables, nothing bundled | classes and objects | 01 |
| Copy-pasted logic for similar models | inheritance | 02 |
| Results that are hard to inspect and combine | special methods | 03 |
| Preprocessing that forgets what it learned (the mystery) | encapsulation, abstract base classes | 04 |
| Code that only works with one kind of model | polymorphism, protocols | 05 |
| Magic numbers, unclear inputs | type hints, docstrings, dataclasses | 07 |
| One big blob doing everything | design and architecture | 08 |
| Silent failures | exceptions and logging | 09 |
| "It worked when I tried it" | testing | 10 |
| Code that only lives in a notebook | packages | 11 |
| All of the above | capstone | 12 |

## Summary

- Code that runs and produces plausible numbers can still be wrong. The worst bugs in data science raise no error.
- Most problems here are not about the model. They are about how the code is organised: what is bundled together, what is repeated, and what remembers what.
- Keep your list. In the capstone (notebook 12) you will turn this notebook into a small, tested package, and the acceptance tests check exactly the bug you just found.

---

## Sources and further reading

**Sources.** This notebook is based on:

- [scikit-learn: Common pitfalls and recommended practices](https://scikit-learn.org/stable/common_pitfalls.html), in particular "Inconsistent preprocessing". This is the bug hidden in this notebook.
- [Sculley et al. (2015), *Hidden Technical Debt in Machine Learning Systems*](https://papers.nips.cc/paper_files/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html), NeurIPS. The classic paper on why ML code rots: glue code, pipeline jungles, configuration debt.

**For self-study.** Where to go next if you want to learn more:

- [Wilson et al. (2017), *Good enough practices in scientific computing*](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1005510), PLOS Computational Biology (paper, beginner): a short, practical checklist written for researchers. Ideal if you come from academia.
- [Joel Grus, *I don't like notebooks*](https://www.youtube.com/watch?v=7jiPeIFXb6U) (talk, JupyterCon 2018, 50 min): an entertaining look at the hidden-state problems of notebooks.
- [Google: Rules of Machine Learning](https://developers.google.com/machine-learning/guides/rules-of-ml) (guide, intermediate): engineering lessons from production ML, including training/serving skew.

---

[← P0 · Python refresher (pre-work)](P0_python_refresher.ipynb) &nbsp;|&nbsp; [Course overview](../README.md) &nbsp;|&nbsp; [01 · Classes and objects →](01_classes_and_objects.ipynb)